# 2021 Census of Population: Dot Density

The census profile presents information from the 2021 Census of Population for various levels of geography, including provinces and territories, census metropolitan areas, communities and census tracts.

This is the continuation of the first notebook. On this notebook all the EDA will be performed. Including spatial analysis.

## Reading the Silver database tables

First, import the dataframe that was previosly created and contains the information of interest.

In [1]:
# import libraries
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon

import duckdb

In [2]:
# show all columns
pd.set_option('display.max_columns', None)

In [3]:
# create a DuckDB database
con = duckdb.connect("data/database/silver.duckdb")


In [4]:
# install spatial
con.execute("INSTALL spatial")
# load spatial
con.execute("LOAD spatial")

In [5]:
# check the tables
con.execute("SHOW TABLES").fetch_df()

,name
0,bdry_da_c2021
1,wide_total_da_c2021


In [8]:
# inspect wide table
con.execute(f"""

    SELECT *   
    FROM wide_total_da_c2021

""").fetch_df()

,DGUID,Canadian,English,Irish,Scottish,"French, n.o.s.",German,Chinese,Italian,Indian (India),Ukrainian,Dutch,Polish,Québécois,"British Isles, n.o.s.",Filipino,French Canadian,"Caucasian (White), n.o.s.","First Nations (North American Indian), n.o.s.",Métis,"European, n.o.s.",Russian,Norwegian,Welsh,Portuguese,American,Spanish,Swedish,Hungarian,Acadian,Pakistani,"African, n.o.s.",Jewish,Punjabi,Vietnamese,"Arab, n.o.s.",Greek,Jamaican,"Asian, n.o.s.","Cree, n.o.s.",Korean,Romanian,Lebanese,Iranian,"Christian, n.i.e.",Danish,"North American Indigenous, n.o.s.",Sikh,Austrian,Belgian,Haitian,Hindu,Mexican,Mennonite,Swiss,Finnish,Sri Lankan,Croatian,Japanese,"South Asian, n.o.s.","Mi'kmaq, n.o.s.","Northern European, n.o.s.",Muslim,Egyptian,"Latin, Central or South American, n.o.s.",Tamil,Icelandic,Colombian,Moroccan,Czech,Syrian,Guyanese,Afghan,"Black, n.o.s.",Serbian,Ojibway,Newfoundlander,Hong Konger,Ontarian,Persian,Trinidadian/Tobagonian,Turkish,"Inuit, n.o.s.",Bangladeshi,Algerian,Brazilian,Nigerian,Armenian,Slovak,"Eastern European, n.o.s.",Somali,Taiwanese,Iraqi,Salvadorean,African Caribbean,"East or Southeast Asian, n.o.s.","West or Central Asian or Middle Eastern, n.o.s.","Caribbean, n.o.s.",Algonquin,"West Indian, n.o.s.",Lithuanian,South African,Australian,Palestinian,Chilean,Congolese,Nova Scotian,Ethiopian,"Hispanic, n.o.s.",Peruvian,Yoruba,Cambodian (Khmer),Berber,Albanian,Maltese,Macedonian,Slovenian,"Western European, n.o.s.",New Brunswicker,Gujarati,Eritrean,African Canadian,Israeli,Mohawk,"Czechoslovakian, n.o.s.",Bulgarian,Albertan,Ghanaian,Barbadian,African American,"Yugoslavian, n.o.s.",Tunisian,"Slavic, n.o.s.",Cuban,Bosnian,Venezuelan,"Innu/Montagnais, n.o.s.",Latvian,Bengali,Cameroonian,Guatemalan,Indonesian,Laotian,Ilocano,Northern Irish,"Celtic, n.o.s.",British Columbian,Ecuadorian,Franco Ontarian,Argentinian,Estonian,Kurdish,Fijian,Jatt,"North American, n.o.s.",Coptic,Thai,Dominican,Nepali,Kabyle,Assyrian,Igbo,Byelorussian,"Dene, n.o.s.","Blackfoot, n.o.s.",Abenaki,Moldovan,"Iroquois (Haudenosaunee), n.o.s.",New Zealander,Sudanese,Breton,Pennsylvania Dutch,Malaysian,Plains Cree,"North African, n.o.s.",Huron (Wendat),Saskatchewanian,Buddhist,Gaspesian,Norman,"Southern or East African, n.o.s.",Ivorian,Saulteaux,"Anishinaabe, n.o.s.",Burundian,Tigrinya,Nicaraguan,Mauritian,Kenyan,Oji-Cree,Vincentian,Jordanian,Manitoban,Cape Bretoner,Rwandan,Grenadian,Malayali,Chaldean,Sinhalese,Mayan,Honduran,Cherokee,Qalipu Mi'kmaq,Indo-Caribbean,Flemish,United Empire Loyalist,Senegalese,Azerbaijani,Sicilian,Pashtun,Malay,Goan,"Bantu, n.o.s.",Tibetan,Zimbabwean,Burmese,Mongolian,Azorean,Atikamekw,Bamileke,Indo-Guyanese,Ugandan,Oromo,Tanzanian,Yemeni,Central African,Libyan,Basque,Uruguayan,"Akan, n.o.s.","Central or West African, n.o.s.",Igorot,Fulani,Woodland Cree,Guinean,St. Lucian,Prince Edward Islander,Maliseet,Beninese,Telugu,Roma,Costa Rican,African Nova Scotian,Malagasy,Kashmiri,Singaporean,Karen,Edo,Tajik,Amhara,Paraguayan
0,2021S051235200380,20.0,30.0,45.0,25.0,0.0,40.0,220.0,20.0,30.0,0.0,0.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,0.0,0.0,0.0,85.0,0.0,0.0,0.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,

In [6]:
# ispect boundaries
con.execute(f"""

    SELECT *   
    FROM bdry_da_c2021

""").fetch_df()

,DGUID,LANDAREA,geometry
0,2021S051235140110,44.2501,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 92, 0, 0, 0, 160, ..."
1,2021S051235140111,37.5066,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 166, 0, 0, 0, 136,..."
2,2021S051235140112,17.5309,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 106, 0, 0, 0, 64, ..."
3,2021S051235140113,18.7260,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 118, 0, 0, 0, 56, ..."
4,2021S051235140114,0.3530,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 39, 0, 0, 0, 136, ..."
...,...,...,...
13125,2021S051235431433,13.6631,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 83, 1, 0, 0, 240, ..."
13126,2021S051235431434,4.1303,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 57, 0, 0, 0, 248, ..."
13127,2021S051235431435,0.8806,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 140, 0, 0, 0, 72, ..."
13128,2021S051235431436,0.6513,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 111, 0, 0, 0, 64, ..."


# Dot Density Map

In [14]:
# inspect wide table
con.execute(f"""

    DESCRIBE wide_total_da_c2021

""").fetch_df()

,column_name,column_type,null,key,default,extra
0,DGUID,VARCHAR,YES,None,None,None
1,Canadian,DOUBLE,YES,None,None,None
2,English,DOUBLE,YES,None,None,None
3,Irish,DOUBLE,YES,None,None,None
4,Scottish,DOUBLE,YES,None,None,None
...,...,...,...,...,...,...
246,Karen,DOUBLE,YES,None,None,None
247,Edo,DOUBLE,YES,None,None,None
248,Tajik,DOUBLE,YES,None,None,None
249,Amhara,DOUBLE,YES,None,None,None


In [ ]:
# recalculate values as 1 per 100 persons with rounded values
#  anything below 50 becomes 0 else becomes 1
factor = 100

con.execute(f"""

    SELECT 
        DGUID,
        ROUND(
            COLUMNS(* EXCLUDE DGUID) / {factor} 
        )
    FROM wide_total_da_c2021;

""").fetch_df()

,DGUID,Canadian,English,Irish,Scottish,"French, n.o.s.",German,Chinese,Italian,Indian (India),Ukrainian,Dutch,Polish,Québécois,"British Isles, n.o.s.",Filipino,French Canadian,"Caucasian (White), n.o.s.","First Nations (North American Indian), n.o.s.",Métis,"European, n.o.s.",Russian,Norwegian,Welsh,Portuguese,American,Spanish,Swedish,Hungarian,Acadian,Pakistani,"African, n.o.s.",Jewish,Punjabi,Vietnamese,"Arab, n.o.s.",Greek,Jamaican,"Asian, n.o.s.","Cree, n.o.s.",Korean,Romanian,Lebanese,Iranian,"Christian, n.i.e.",Danish,"North American Indigenous, n.o.s.",Sikh,Austrian,Belgian,Haitian,Hindu,Mexican,Mennonite,Swiss,Finnish,Sri Lankan,Croatian,Japanese,"South Asian, n.o.s.","Mi'kmaq, n.o.s.","Northern European, n.o.s.",Muslim,Egyptian,"Latin, Central or South American, n.o.s.",Tamil,Icelandic,Colombian,Moroccan,Czech,Syrian,Guyanese,Afghan,"Black, n.o.s.",Serbian,Ojibway,Newfoundlander,Hong Konger,Ontarian,Persian,Trinidadian/Tobagonian,Turkish,"Inuit, n.o.s.",Bangladeshi,Algerian,Brazilian,Nigerian,Armenian,Slovak,"Eastern European, n.o.s.",Somali,Taiwanese,Iraqi,Salvadorean,African Caribbean,"East or Southeast Asian, n.o.s.","West or Central Asian or Middle Eastern, n.o.s.","Caribbean, n.o.s.",Algonquin,"West Indian, n.o.s.",Lithuanian,South African,Australian,Palestinian,Chilean,Congolese,Nova Scotian,Ethiopian,"Hispanic, n.o.s.",Peruvian,Yoruba,Cambodian (Khmer),Berber,Albanian,Maltese,Macedonian,Slovenian,"Western European, n.o.s.",New Brunswicker,Gujarati,Eritrean,African Canadian,Israeli,Mohawk,"Czechoslovakian, n.o.s.",Bulgarian,Albertan,Ghanaian,Barbadian,African American,"Yugoslavian, n.o.s.",Tunisian,"Slavic, n.o.s.",Cuban,Bosnian,Venezuelan,"Innu/Montagnais, n.o.s.",Latvian,Bengali,Cameroonian,Guatemalan,Indonesian,Laotian,Ilocano,Northern Irish,"Celtic, n.o.s.",British Columbian,Ecuadorian,Franco Ontarian,Argentinian,Estonian,Kurdish,Fijian,Jatt,"North American, n.o.s.",Coptic,Thai,Dominican,Nepali,Kabyle,Assyrian,Igbo,Byelorussian,"Dene, n.o.s.","Blackfoot, n.o.s.",Abenaki,Moldovan,"Iroquois (Haudenosaunee), n.o.s.",New Zealander,Sudanese,Breton,Pennsylvania Dutch,Malaysian,Plains Cree,"North African, n.o.s.",Huron (Wendat),Saskatchewanian,Buddhist,Gaspesian,Norman,"Southern or East African, n.o.s.",Ivorian,Saulteaux,"Anishinaabe, n.o.s.",Burundian,Tigrinya,Nicaraguan,Mauritian,Kenyan,Oji-Cree,Vincentian,Jordanian,Manitoban,Cape Bretoner,Rwandan,Grenadian,Malayali,Chaldean,Sinhalese,Mayan,Honduran,Cherokee,Qalipu Mi'kmaq,Indo-Caribbean,Flemish,United Empire Loyalist,Senegalese,Azerbaijani,Sicilian,Pashtun,Malay,Goan,"Bantu, n.o.s.",Tibetan,Zimbabwean,Burmese,Mongolian,Azorean,Atikamekw,Bamileke,Indo-Guyanese,Ugandan,Oromo,Tanzanian,Yemeni,Central African,Libyan,Basque,Uruguayan,"Akan, n.o.s.","Central or West African, n.o.s.",Igorot,Fulani,Woodland Cree,Guinean,St. Lucian,Prince Edward Islander,Maliseet,Beninese,Telugu,Roma,Costa Rican,African Nova Scotian,Malagasy,Kashmiri,Singaporean,Karen,Edo,Tajik,Amhara,Paraguayan
0,2021S051235200380,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0

In [18]:
# function with preserve sampling size column name
def random_pts(gdf, start_col_index, end_col_index):

    col_list = list(range(start_col_index,end_col_index)) # define list with required range
    all_points = [] # define empty list
    
    for n in col_list:

        # sampling points within geom
        multipoints = gdf.sample_points(size=gdf.iloc[:,n])
        
        # making a GeoDataFrame
        mp_gdf = gpd.GeoDataFrame(
            geometry = multipoints,
            crs=gdf.crs
        )
    
        mp_gdf['source'] = gdf.columns[n]
    
        # make a dataframe with each random point
        points_gdf = mp_gdf.explode(index_parts=False).reset_index(drop=True)

        # add to the list
        all_points.append(points_gdf)

    # concatenate all points 
    result = pd.concat(all_points, ignore_index=True)

    return result

In [19]:
%%time
points_gdf = random_pts(gdf32_c1,1,251)
points_gdf.head()

CPU times: total: 42min 43s
Wall time: 43min 27s


,source,geometry
0,Canadian,POINT (7342880.31 1002511.515)
1,Canadian,POINT (7348691.216 1006013.541)
2,Canadian,POINT (7344159.083 1010397.551)
3,Canadian,POINT (7337720.988 1017356.652)
4,Canadian,POINT (7340670.202 1016882.431)


In [20]:
# Write the results to file.
points_gdf.to_csv('dd_ethnic_origin.csv', index=False)

### Geographical IDs

Insert Description

# Next Steps

There are two other subtopics related to the inmigrant population that may be included in this analysis:

- **Place of birth for the recent immigrant population in private household**. It would be interesting to compare the numbers between recent inmgrants and the overall inmigrant population to see if there are significant changes.
- **Ethnic or cultural origin for the population in private households**. Analysing the population by cultural origin would also provide an interesting perspective, as culture and country of origin may overlap but are not necessarily equivalent.

In [ ]:
# splitting the df by subtopic and characteristic

#  group 2: Place of birth for the recent immigrant population in private households
#df2=df[df["CHARACTERISTIC_ID"].between(1604,1664)].reset_index().drop(columns=["CHARACTERISTIC_ID","index"])

# Resouces

- [Census Profile, 2021 Census of Population](https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/index.cfm?Lang=E)
- [About the Census Profile, 2021 Census of Population](https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/about-apropos/about-apropos.cfm?Lang=E#aa1)
- [Guide to the Census of Population, 2021](https://www12.statcan.gc.ca/census-recensement/2021/ref/98-304/index-eng.cfm),  provides an overview of the Census of Population content determination, collection, processing, data quality assessment and data dissemination. It may be useful to both new and experienced users who wish to familiarize themselves with and find specific information about the 2021 Census
- [Filling the gaps: Information on gender in the 2021 Census](https://www12.statcan.gc.ca/census-recensement/2021/ref/98-20-0001/982000012021001-eng.cfm), defines gender, sex at birth, and relevant concepts as the Census 2021 disseminates census information on gender
- [Census Profile metadata](https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/about-apropos/metadata-metadonnees-eng.cfm), list characteristics by topics and subtopic, and list all footnotes
- [Full Table Download (CSV) User Guide](https://www.statcan.gc.ca/en/developers/csv/user-guide), provides users with a guide to the full table downloadable output files available from the Statistics Canada website
- [Dictionary, Census of Population 2021, PDF version](https://www12.statcan.gc.ca/census-recensement/2021/ref/dict/98-301-x2021001-eng.pdf), is a reference document which contains detailed definitions of Census of Population concepts, variables and geographic terms, as well as historical information. The PDF version organizes the concepts by topics, which is not the case for the [web version](https://www12.statcan.gc.ca/census-recensement/2021/ref/dict/index-eng.cfm).
- [2021 Census – Boundary files](https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/index2021-eng.cfm?year=21)
- [Boundary Files, Reference Guide, Census year 2021, Second Edition](https://www150.statcan.gc.ca/n1/en/catalogue/92-160-G2021002)

# Reference

Statistics Canada. 2023. Census Profile. 2021 Census of Population. Statistics Canada Catalogue number 98-316-X2021001. Ottawa. Released November 15, 2023.
*https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/index.cfm?Lang=E (accessed August 4, 2025).*